# Mast Cell Pipeline Validation

## Scope

This notebook validates the **CellQuorum backbone pipeline** on the published lymphedema mast-cell cohort:

- **CellRanger ingestion** → **SoupX ambient correction** → **QC** → **normalization** → **PCA** → **Harmony integration** → **Leiden clustering** → **marker-vote annotation**

It checks:
1. The pipeline executes end-to-end on real 10x data
2. The SoupX **litmus test**: mast-cell markers remain high (~80-85%), ECM contaminants drop low (~2-6%)
3. Broad structural anchors: ~79k cells post-QC, ~10 cell types, ~3,992 mast cells, Harmony iLISI ~2.2→3.0

This notebook does **NOT** perform:
- CellTypist labels (planned downstream stage)
- Mast cell substate analysis (planned downstream stage)
- Pseudobulk differential expression (planned downstream stage)

Those are post-annotation stages that will be added after the backbone is locked.

In [ ]:
# Smoke test: 2-library run (first Normal+LE pair)
import shutil
import tempfile
from pathlib import Path

import pandas as pd

from cellquorum import run_pipeline

# Guard: check if CellRanger data and R/SoupX are available
data_root = Path("/mnt/e/lymphedema_cellranger")
rscript_available = shutil.which("Rscript") is not None

if not data_root.exists() or not rscript_available:
    print("SKIPPING: This cell requires /mnt/e/lymphedema_cellranger and R + SoupX.")
    print(f"  Data root exists: {data_root.exists()}")
    print(f"  Rscript available: {rscript_available}")
    result = None
else:
    # Build a 2-library manifest (first Normal+LE pair: Set4_norm4_v8 + Set4_LE4_v8)
    full_manifest = pd.read_csv("../configs/manifests/mast_cell_5patient.csv")
    smoke_manifest = full_manifest.head(2)  # First 2 rows

    # Write the smoke manifest to a temp file
    temp_dir = Path(tempfile.mkdtemp())
    smoke_manifest_path = temp_dir / "smoke_manifest.csv"
    smoke_manifest.to_csv(smoke_manifest_path, index=False)

    # Override the manifest in the config dict
    # (We need to rebuild the config pointing at the smoke manifest)
    from cellquorum.config.loader import load_config

    config = load_config("../configs/mast_cell.yaml")
    config_dict = config.model_dump()
    config_dict["paths"]["manifest"] = str(smoke_manifest_path)
    config_dict["paths"]["output_dir"] = str(temp_dir / "smoke_run")

    # Run the pipeline with the modified config
    result = run_pipeline(config=config_dict)

    print(f"Smoke run complete. Output: {result.context.paths.root}")
    print(f"Total cells: {result.context.adata.n_obs}")
    print(f"Total genes: {result.context.adata.n_vars}")

In [ ]:
# SoupX litmus test: check mast marker retention vs ECM contaminant removal
if result is not None:
    import pandas as pd

    adata = result.context.adata

    # Subset to mast cells
    mast_cells = adata[adata.obs["cell_type"] == "Mast cells"].copy()

    # Get the counts layer (SoupX-corrected counts)
    counts = mast_cells.layers["counts"]

    # Mast markers: TPSAB1, CPA3, KIT
    mast_markers = ["TPSAB1", "CPA3", "KIT"]
    # ECM contaminants: COL1A1, COL1A2, COL3A1, LUM, DCN
    ecm_markers = ["COL1A1", "COL1A2", "COL3A1", "LUM", "DCN"]

    # Compute detection fractions
    def detection_fraction(adata_subset, gene_list) -> dict[str, float | None]:  # noqa: ANN001
        """Compute fraction of cells with >0 counts for each gene."""
        results = {}
        for gene in gene_list:
            if gene in adata_subset.var_names:
                gene_idx = list(adata_subset.var_names).index(gene)
                detected = (counts[:, gene_idx] > 0).sum()
                results[gene] = detected / adata_subset.n_obs
            else:
                results[gene] = None  # Gene not in data
        return results

    mast_detection = detection_fraction(mast_cells, mast_markers)
    ecm_detection = detection_fraction(mast_cells, ecm_markers)

    # Build summary table
    litmus_table = pd.DataFrame(
        {
            "Gene": mast_markers + ecm_markers,
            "Category": ["Mast marker"] * len(mast_markers)
            + ["ECM contaminant"] * len(ecm_markers),
            "Detection %": [
                f"{v*100:.1f}%" if v is not None else "N/A"
                for v in list(mast_detection.values()) + list(ecm_detection.values())
            ],
        }
    )

    print(f"\nSoupX Litmus Test (n={mast_cells.n_obs} mast cells):")
    print(litmus_table.to_string(index=False))
    print("\nExpected at full cohort: mast markers ~80-85%, ECM contaminants ~2-6%.")
    print("On this 2-library smoke test, exact percentages are indicative, not the full anchor.")
else:
    print("Skipped (no result from smoke run).")

## Full 10-Library Run

To validate the pipeline on the complete 5-patient cohort (10 libraries: 5 Normal + 5 LE), run:

```python
from cellquorum import run_pipeline

result = run_pipeline(
    config="../configs/mast_cell.yaml",
    output_dir="../runs/mast_full"
)
```

### Expected Anchors

- **Total cells post-QC**: ~79,000
- **Cell types**: ~10 (Fibroblasts, Keratinocytes, Melanocytes, Endothelial, Pericytes, Mast cells, Macrophages, Dendritic cells, CD4 T cells, CD8 T cells)
- **Mast cells**: ~3,992
- **Harmony iLISI**: ~2.2 (pre-integration) → ~3.0 (post-integration)

### Output Locations

- **Provenance artifacts**: `../runs/mast_full/provenance/`
  - `resolved_config.json`: validated runtime configuration
  - `pipeline_plan.json`: full stage plan + backend status
  - `stage_execution_records.json`: lifecycle records for each stage
  - `artifact_manifest.json`: inventory of all outputs

- **Figures**: `../runs/mast_full/figures/`
  - QC violin plots (genes/counts/mito per cell)
  - PCA variance ratio + elbow
  - Harmony batch mixing metrics
  - Leiden UMAP colored by cluster + cell type

- **Final object**: `../runs/mast_full/objects/adata_final.h5ad`
  - SoupX-corrected counts in `layers['counts']`
  - Normalized data in `.X`
  - PCA in `obsm['X_pca']`
  - Harmony in `obsm['X_pca_harmony']`
  - Cell-type annotations in `obs['cell_type']`